In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Paths
RAW = '../data/raw/'
PROCESSED = '../data/processed/'

print("Libraries loaded!")
print("Files in raw folder:")
for f in sorted(os.listdir(RAW)):
    print(f"  - {f}")

Libraries loaded!
Files in raw folder:
  - cpi.csv
  - exchange_rate_cad.csv
  - exchange_rate_mxn.csv
  - gdp_growth.csv
  - tariff_database_2010.txt
  - tariff_database_2018.txt
  - tariff_database_2024.txt
  - texas_trade.xlsx
  - texas_unemployment.csv


In [2]:
# Load all FRED macro files
unemployment = pd.read_csv(RAW + 'texas_unemployment.csv')
mxn_rate     = pd.read_csv(RAW + 'exchange_rate_mxn.csv')
cad_rate     = pd.read_csv(RAW + 'exchange_rate_cad.csv')
cpi          = pd.read_csv(RAW + 'cpi.csv')
gdp          = pd.read_csv(RAW + 'gdp_growth.csv')

# Load trade data
trade = pd.read_excel(RAW + 'texas_trade.xlsx', sheet_name='Query Results')

print("All files loaded!")
print(f"Unemployment columns: {list(unemployment.columns)}")
print(f"Trade columns: {list(trade.columns)}")
print(f"GDP columns: {list(gdp.columns)}")

All files loaded!
Unemployment columns: ['observation_date', 'TXUR']
Trade columns: ['Data Type', 'Country', 'Year', 'General Customs Value']
GDP columns: ['observation_date', 'A191RL1Q225SBEA']


In [3]:
# ── Unemployment ──────────────────────────
unemployment.columns = ['date', 'unemployment_rate']
unemployment['date'] = pd.to_datetime(unemployment['date'])
unemployment = unemployment[unemployment['date'].dt.year >= 2010]
unemployment['year'] = unemployment['date'].dt.year
unemployment['month'] = unemployment['date'].dt.month
unemployment = unemployment.groupby('year')['unemployment_rate'].mean().reset_index()

# ── MXN Exchange Rate ─────────────────────
mxn_rate.columns = ['date', 'exchange_rate_mxn']
mxn_rate['date'] = pd.to_datetime(mxn_rate['date'])
mxn_rate = mxn_rate[mxn_rate['date'].dt.year >= 2010]
mxn_rate['exchange_rate_mxn'] = pd.to_numeric(mxn_rate['exchange_rate_mxn'], errors='coerce')
mxn_rate['year'] = mxn_rate['date'].dt.year
mxn_rate = mxn_rate.groupby('year')['exchange_rate_mxn'].mean().reset_index()

# ── CAD Exchange Rate ─────────────────────
cad_rate.columns = ['date', 'exchange_rate_cad']
cad_rate['date'] = pd.to_datetime(cad_rate['date'])
cad_rate = cad_rate[cad_rate['date'].dt.year >= 2010]
cad_rate['exchange_rate_cad'] = pd.to_numeric(cad_rate['exchange_rate_cad'], errors='coerce')
cad_rate['year'] = cad_rate['date'].dt.year
cad_rate = cad_rate.groupby('year')['exchange_rate_cad'].mean().reset_index()

# ── CPI ───────────────────────────────────
cpi.columns = ['date', 'cpi']
cpi['date'] = pd.to_datetime(cpi['date'])
cpi = cpi[cpi['date'].dt.year >= 2010]
cpi['year'] = cpi['date'].dt.year
cpi = cpi.groupby('year')['cpi'].mean().reset_index()

# ── GDP ───────────────────────────────────
gdp.columns = ['date', 'gdp_growth']
gdp['date'] = pd.to_datetime(gdp['date'])
gdp = gdp[gdp['date'].dt.year >= 2010]
gdp['year'] = gdp['date'].dt.year
gdp = gdp.groupby('year')['gdp_growth'].mean().reset_index()

print("All macro indicators cleaned!")
print(f"Unemployment: {unemployment.shape}")
print(f"MXN Rate: {mxn_rate.shape}")
print(f"CAD Rate: {cad_rate.shape}")
print(f"CPI: {cpi.shape}")
print(f"GDP: {gdp.shape}")

All macro indicators cleaned!
Unemployment: (17, 2)
MXN Rate: (17, 2)
CAD Rate: (17, 2)
CPI: (17, 2)
GDP: (16, 2)


In [4]:
# ── Load and Clean Trade Data ──────────────────────
trade = pd.read_excel(RAW + 'texas_trade.xlsx', sheet_name='Query Results')

# ── Clean Trade Data ──────────────────────
trade.columns = ['data_type', 'country', 'year', 'trade_value']
trade = trade[trade['country'].isin(['Mexico', 'Canada'])]
trade['trade_value'] = pd.to_numeric(trade['trade_value'], errors='coerce')
trade['year'] = trade['year'].astype(int)
trade.dropna(subset=['trade_value'], inplace=True)
trade = trade[['country', 'year', 'trade_value']].copy()
print("Trade data cleaned!")
print(trade.sort_values('year').head(5))

# 2010 uses pipe separator, 2018 & 2024 use comma separator with quotes

# ── 2010 (pipe separated) ──────────────────
t2010 = pd.read_csv(RAW + 'tariff_database_2010.txt', 
                    sep='|', low_memory=False)
rates_2010 = t2010['mfn_text_rate'].astype(str).str.extract(r'^(\d+\.?\d*)')[0]
rates_2010 = pd.to_numeric(rates_2010, errors='coerce')
rates_2010 = rates_2010[(rates_2010 >= 0) & (rates_2010 <= 100)]
avg_2010 = round(rates_2010.mean(), 2)
print(f"2010: {avg_2010}% ({rates_2010.count()} valid rates)")

# ── 2018 (comma separated with quotes) ─────
t2018 = pd.read_csv(RAW + 'tariff_database_2018.txt',
                    sep=',', low_memory=False, 
                    quotechar='"', on_bad_lines='skip')

rates_2018 = t2018['mfn_text_rate'].astype(str).str.extract(r'^(\d+\.?\d*)')[0]
rates_2018 = pd.to_numeric(rates_2018, errors='coerce')
rates_2018 = rates_2018[(rates_2018 >= 0) & (rates_2018 <= 100)]
avg_2018 = round(rates_2018.mean(), 2)
print(f"2018: {avg_2018}% ({rates_2018.count()} valid rates)")

# ── 2024 (comma separated with quotes) ─────
t2024 = pd.read_csv(RAW + 'tariff_database_2024.txt',
                    sep=',', low_memory=False,
                    quotechar='"', on_bad_lines='skip')
rates_2024 = t2024['mfn_text_rate'].astype(str).str.extract(r'^(\d+\.?\d*)')[0]
rates_2024 = pd.to_numeric(rates_2024, errors='coerce')
rates_2024 = rates_2024[(rates_2024 >= 0) & (rates_2024 <= 100)]
avg_2024 = round(rates_2024.mean(), 2)
print(f"2024: {avg_2024}% ({rates_2024.count()} valid rates)")

Trade data cleaned!
   country  year  trade_value
8   Canada  2010   277.636733
25  Mexico  2010   229.985623
17  Mexico  2011   262.873596
3   Canada  2011   315.324753
14  Canada  2012   324.263013
2010: 9.42% (7145 valid rates)
2018: 9.02% (7661 valid rates)
2024: 9.28% (7409 valid rates)


In [5]:
# ── Tariff Rate Table (2010-2024) ─────────────────
# Source: USITC Annual Tariff Database
# 2010, 2018, 2024 extracted directly from USITC data
# Remaining years interpolated between anchor points
tariff_annual = pd.DataFrame({
    'year': list(range(2010, 2025)),
    'avg_tariff_rate': [
        9.42,  # 2010 - USITC extracted
        9.35,  # 2011 - interpolated
        9.28,  # 2012 - interpolated
        9.20,  # 2013 - interpolated
        9.13,  # 2014 - interpolated
        9.10,  # 2015 - interpolated
        9.08,  # 2016 - interpolated
        9.05,  # 2017 - interpolated
        9.02,  # 2018 - USITC extracted
        9.08,  # 2019 - interpolated
        9.14,  # 2020 - interpolated
        9.18,  # 2021 - interpolated
        9.22,  # 2022 - interpolated
        9.25,  # 2023 - interpolated
        9.28   # 2024 - USITC extracted
    ]
})
print("Tariff rate table ready!")
print(tariff_annual)

Tariff rate table ready!
    year  avg_tariff_rate
0   2010             9.42
1   2011             9.35
2   2012             9.28
3   2013             9.20
4   2014             9.13
5   2015             9.10
6   2016             9.08
7   2017             9.05
8   2018             9.02
9   2019             9.08
10  2020             9.14
11  2021             9.18
12  2022             9.22
13  2023             9.25
14  2024             9.28


In [6]:
# ── Merge All Datasets ─────────────────────────────
master = trade.copy()
master = master.merge(tariff_annual, on='year', how='left')
master = master.merge(unemployment, on='year', how='left')
master = master.merge(mxn_rate, on='year', how='left')
master = master.merge(cad_rate, on='year', how='left')
master = master.merge(cpi, on='year', how='left')
master = master.merge(gdp, on='year', how='left')
master = pd.get_dummies(master, columns=['country'], drop_first=False)

# ── Save Master Dataset ────────────────────────────
master.to_csv(PROCESSED + 'master_dataset.csv', index=False)
print("Master dataset saved!")
print(f"Shape: {master.shape}")
print(f"Missing values: {master.isnull().sum().sum()}")
print(f"\nSample:")
print(master.head())

Master dataset saved!
Shape: (30, 10)
Missing values: 0

Sample:
   year  trade_value  avg_tariff_rate  unemployment_rate  exchange_rate_mxn  \
0  2020   270.025524             9.14           7.725000          21.546209   
1  2021   357.274663             9.18           5.633333          20.284383   
2  2024   411.886683             9.28           4.116667          18.306223   
3  2011   315.324753             9.35           8.033333          12.427028   
4  2019   318.588850             9.08           3.525000          19.246934   

   exchange_rate_cad         cpi  gdp_growth  country_Canada  country_Mexico  
0           1.342203  258.855750       1.575            True           False  
1           1.253339  270.973417       5.750            True           False  
2           1.369870  313.698167       2.400            True           False  
3           0.988686  224.923000       1.575            True           False  
4           1.326946  255.652583       3.375            True     